# Практическое долгосрочное задание 1
## 👄 Интерполяция формы губ по видео



## ЧАСТЬ 0. 🎯 Введение

Нам в руки попали **обрывки данных** — редкие снимки губ человека, произносящего звуки **восточноевропейского алфавита** 🕵️

Наш агент, внедрённый глубоко под прикрытием, снимал его **раз в секунду** — строго по **равномерной временно́й сетке**. Никаких отклонений, никаких дополнительных замеров. Только сухие факты: кадр, ещё кадр, и снова тишина.




Между этими кадрами — **пустота**. Звуки шли один за другим, но мы видим лишь **редкие вспышки мимики**: губы то округляются, то растягиваются, то сжимаются — и всё это **урывками**, без плавных переходов. Мимика кажется **рваной**, движения — **дёргаными**. Смотреть на такое больно 😖

---

### 🚀 Наша финальная задача

#### Научиться воспроизводить **гладкую анимацию** для… 🌀

…впрочем, узнаем **потом**. 😏

---

### 🎬 А пока — начнём с малого!

#### Попробуем восстановить хотя бы **отдельные звуки** 🗣️

<table>
<tr>
<td align="center">👄</td>
<td align="center">🎞️</td>
<td align="center">✨</td>
<td align="center">🔊</td>
</tr>
<tr>
<td align="center">губы</td>
<td align="center">кадры</td>
<td align="center">сгладим</td>
<td align="center">зазвучит!</td>
</tr>
</table>

---

> 💭 **Шаг за шагом:**
> 1. 📍 Соберём редкие кадры
> 2. 🧮 Построим интерполяцию
> 3. 🎥 Оживим анимацию
> 4. 🌍 А потом — уже что-то большее…

In [88]:
#Импорт нужных библиотек

from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import glob
from collections import defaultdict
from draw import *

import pickle

In [91]:
#Загрузка данных
with open('LipsArrayHalfTime.np' , 'rb') as f:
    LipsData = pickle.load(f)

### 📦 Структура данных: `LipsData`

`LipsData` — это **словарь**, ставящий в соответствие каждой букве её временну́ю траекторию движения губ:

$$
\text{LipsData} : \quad \text{буква} \;\longmapsto\; \mathbf{X} \in \mathbb{R}^{n \times 40 \times 2}
$$

---

#### 📐 Что означает каждый индекс

$$
\mathbf{X}[t,\, p,\, c], \qquad
\begin{aligned}
&t = 0, \dots, n-1 \\
&p = 0, \dots, 39 \\
&c \in \{0, 1\}
\end{aligned}
$$

| Индекс | Обозначение | Смысл |
|:---:|:---:|:---|
| $t$ | **время** | номер такта — дискретный отсчёт вдоль оси времени |
| $p$ | **точка** | 40 ключевых точек контура губ (20 внешних + 20 внутренних) |
| $c$ | **координата** | $c=0 \Rightarrow x$, $\;c=1 \Rightarrow y$ на плоскости |

---

#### 🕰️ О такте

> **Такт** — это условная единица времени.  
> Можно считать, что $1$ такт $\approx 0.1$ с, но эта привязка несущественна:  
> для интерполяции важна лишь **последовательность** отсчётов, а не их физическая длительность.

---

#### 🧠 Как это читать

Для фиксированной буквы $\ell$ имеем:

$$
\text{LipsData}[\ell] \;=\; \bigl\{\, \mathbf{x}_t \,\bigr\}_{t=0}^{\,n-1},
\qquad
\mathbf{x}_t \in \mathbb{R}^{40 \times 2}
$$

То есть каждая буква — это **растянутая во времени последовательность** «снимков» формы губ:

$$
\mathbf{x}_0 \;\longrightarrow\; \mathbf{x}_1 \;\longrightarrow\; \cdots \;\longrightarrow\; \mathbf{x}_{n-1}
$$

где каждый $\mathbf{x}_t$ — плоская конфигурация из 40 точек:

$$
\mathbf{x}_t \;=\;
\bigl\{\, (x_p,\, y_p) \,\bigr\}_{p=0}^{39}
$$


### 🎬 Визуализация: функция `animate_lips`

Преподаватель предоставил **готовую функцию** для интерактивной визуализации:

```python
animate_lips(sequence, interval=...)
```
Предлагается оставить значение интервала равным 200. Однако во время интерполяции желательно увеличивать это время пропорционально отнощению желанного количества отрисованных кадров к исходному количеству кадров
### Ниже представлен пример для буквы А

In [94]:
animate_lips(LipsData['А'], interval=200)

## 📐 Часть 1. Интерполяционный многочлен Лагранжа


#### НАПОМНИМ

Пусть даны $n$ узлов $t_0 < t_1 < \dots < t_{n-1}$ и значения функции в них $y_0, y_1, \dots, y_{n-1}$.

**Задача:** найти многочлен $P(t)$ степени $\le n-1$, который **точно проходит через все узлы**:

$$
P(t_i) = y_i, \qquad i = 0, \dots, n-1
$$

---

#### 🧮 Формула Лагранжа

Решение единственно и записывается в виде

$$
P(t) \;=\; \sum_{i=0}^{n-1} y_i \, L_i(t)
$$

где **базисные многочлены Лагранжа**:

$$
L_i(t) \;=\; \prod_{\substack{j=0 \\ j \ne i}}^{n-1} \frac{t - t_j}{\,t_i - t_j\,}
$$

---

#### ✨ Ключевое свойство

$$
L_i(t_k) \;=\;
\begin{cases}
1, & i = k \\[4pt]
0, & i \ne k
\end{cases}
$$

То есть каждый базисный многочлен «отвечает» **только за свой узел**.  
Отсюда мгновенно следует:

$$
P(t_k) = \sum_i y_i \, L_i(t_k) = y_k \quad ✔
$$

---



#### 🤔 Что же делать в части 1 ???

Мы применим эту формулу к **каждой из 80 координат** наших губ:

$$
(x_0, y_0),\;(x_1, y_1),\;\dots,\;(x_{39}, y_{39})
$$

и восстановим **пропущенные кадры** между известными узлами.

#########################################################################################################################

### 🎯 Задание

Дан словарь `LipsData`, где:

- 🔑 ключ — буква,
- 📊 значение — массив `(n, 40, 2)`.

**Требуется:**

1. Для каждой буквы $\ell$ построить **интерполяционный многочлен Лагранжа**
   для каждой из $40 \times 2 = 80$ координат.

   
   В качестве узлов использовать **все исходные кадры**:

   $$
   t_i = 0,\; 1,\; 2,\; \dots,\; n-1
   $$

3. Вычислить значения многочлена на **вдвое более плотной сетке**:

   $$
   t = 0,\; 0.5,\; 1,\; 1.5,\; 2,\; 2.5,\; \dots,\; n-1
   $$

   То есть кадров станет **в 2 раза больше**, чем было изначально.

4. Собрать результат в массив формы `(2n − 1, 40, 2)`.

5. **Визуализировать** результат функцией `animate_lips`.

---

#### ✅ Что должно получиться

- 📈 Каждая буква — восстановленная последовательность `(n, 40, 2)`.
- 🎬 Плавная анимация, где промежуточные кадры заполнены.



In [ ]:
##TODO
##ЗДЕСЬ ДОЛЖЕН БЫТЬ ВАШ КОД
## Результат запишите в переменную Task1LispPoints, представляющую собой 
## словарь , где ключ - буква , а значение - массив размера (n, 40, 2) (значение 2n − 1 зависит от буквы)


#SOME CODE




#Визуализация
animate_lips(Task1LispPoints['В'], interval=100)

### ⚠️ Внимание!

> **Не ждите, что результаты будут прям красивыми** 😅  


> 🧠 **Если что-то не получается — это нормально!**  
> Подумайте, **почему** результат выглядит именно так,  
> и выпишите свои мысли в ячейке ниже. ✍️

#### ✍️ Запишите здесь, что вы наблюдаете и почему, по вашему мнению,
####    результат выглядит именно так.
\#
    ВАШИ МЫСЛИ
\#

In [ ]:
#########################################################################################################################

## 📐 Часть 2. СКОРО БУДЕТ!!!